<center><h1>Heterogeneous Graph (<em>Heterograph</em>) Construction for TCGA-BRCA</h1></center></br>


<p>A <strong>Heterogeneous Graph</strong> (<em>Heterograph</em>) contains multiple types of nodes and/or edges. Unlike homogeneous graphs where all elements are treated uniformly, heterogeneous graphs model diverse entities and relationships explicitly.</p>

<div>
  <b>Heterogeneous vs. Homogeneous Graphs:</b>
  <table>
    <thead>
      <tr>
        <th>Feature</th>
        <th>Homogeneous Graph</th>
        <th>Heterogeneous Graph</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td>Node types</td>
        <td>Single type</td>
        <td>Multiple types (gene, protein, case)</td>
      </tr>
      <tr>
        <td>Edge types</td>
        <td>Single relation</td>
        <td>Multiple relations (interaction, similarity)</td>
      </tr>
      <tr>
        <td>Representation simplicity</td>
        <td>Simpler</td>
        <td>Richer, more flexible</td>
      </tr>
      <tr>
        <td>Expressiveness</td>
        <td>Limited</td>
        <td>Higher (captures real-world complexity)</td>
      </tr>
    </tbody>
  </table>
</div>
    
  <p><b> Heterogeneous Graphs for Multi-Omics Data:</b></p>
  <ul>
    <li>Represent diverse omics layers as different node types.</li>
    <li>Capture cross-omics interactions (gene–protein, case–mutation).</li>
    <li>Preserve biological structure and semantics for deeper insights.</li>
  </ul>
  <p>Homogeneous graphs flatten this structure and lose critical biological context.</p>



In [ ]:
# !pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q torch-sparse -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q torch-cluster -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:
import torch
from torch_geometric.data import HeteroData
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import pandas as pd
import numpy as np
import json
import os
from collections import OrderedDict

In [ ]:
dataset_keys = {
    "tcga_brca_data", "gene_data", "protein_data",
    "cnv_data", "mutation_data", "status_data", "subtype_data"
}

for key in dataset_keys:
    globals()[key] = None

def set_source_datasets(preprocessed_df):
    for key in dataset_keys:
        if key in preprocessed_df:
            globals()[key] = preprocessed_df[key]
        else:
            raise KeyError(f"Missing key '{key}' in preprocessed_df")

In [ ]:
subtypes_list = []
patient_features = None
subtype_tensor = None
gene_names = []
protein_names = []
cnv_names = []
mutation_names = []
gene_to_idx = {}
prot_to_idx = {}
cnv_to_idx = {}
mutation_to_idx = {}

In [ ]:
biological_mappings = {}
mappings_save_dir = 'graph_mappings'

def create_persistent_mappings():
    """Create deterministic biological mappings that are consistent across runs"""
    global biological_mappings
    
    print("Creating deterministic biological mappings...")
    os.makedirs(mappings_save_dir, exist_ok=True)
    
    biological_mappings = {
        'gene': OrderedDict(),
        'protein': OrderedDict(),
        'cnv': OrderedDict(),
        'mutation': OrderedDict(),
        'subtype': OrderedDict(),
        'case': OrderedDict()
    }
    
    # 1. SUBTYPE MAPPINGS
    print("Creating deterministic subtype mappings...")
    sorted_subtypes = sorted(subtype_data.columns.tolist())  # Alphabetical sort for consistency
    for idx, subtype_name in enumerate(sorted_subtypes):
        biological_mappings['subtype'][idx] = subtype_name
    
    # 2. GENE MAPPINGS
    sorted_gene_cols = sorted(gene_data.columns.tolist())
    for idx, col_name in enumerate(sorted_gene_cols):
        bio_name = col_name.split('_', 1)[1] if '_' in col_name else col_name
        biological_mappings['gene'][idx] = bio_name
    
    # 3. PROTEIN MAPPINGS  
    sorted_protein_cols = sorted(protein_data.columns.tolist())
    for idx, col_name in enumerate(sorted_protein_cols):
        bio_name = col_name.split('_', 1)[1] if '_' in col_name else col_name
        biological_mappings['protein'][idx] = bio_name
    
    # 4. CNV MAPPINGS
    sorted_cnv_cols = sorted(cnv_data.columns.tolist())
    for idx, col_name in enumerate(sorted_cnv_cols):
        bio_name = col_name.split('_', 1)[1] if '_' in col_name else col_name
        biological_mappings['cnv'][idx] = bio_name
    
    # 5. MUTATION MAPPINGS
    sorted_mutation_cols = sorted(mutation_data.columns.tolist())
    for idx, col_name in enumerate(sorted_mutation_cols):
        bio_name = col_name.split('_', 1)[1] if '_' in col_name else col_name
        biological_mappings['mutation'][idx] = bio_name
    
    # 6. CASE MAPPINGS
    for idx in range(len(tcga_brca_data)):
        patient_id = f"Patient_{idx:04d}"  # Consistent patient IDs
        biological_mappings['case'][idx] = patient_id
    
    print("\nMapping summary:")
    for node_type, mapping in biological_mappings.items():
        print(f"    {node_type}: {len(mapping)} nodes")
    
    # Save mappings to files
    save_biological_mappings()


In [ ]:
def save_biological_mappings():
    """Save biological mappings to JSON files for later use"""
    print("\nSaving biological mappings to files...")
    
    # Convert OrderedDict to regular dict for JSON serialization
    mappings_to_save = {}
    for node_type, mapping in biological_mappings.items():
        mappings_to_save[node_type] = dict(mapping)
    
    # Save main mappings file
    main_file = os.path.join(mappings_save_dir, 'biological_mappings.json')
    with open(main_file, 'w') as f:
        json.dump(mappings_to_save, f, indent=2)
    
    # Save individual mapping files for convenience
    for node_type, mapping in mappings_to_save.items():
        individual_file = os.path.join(mappings_save_dir, f'{node_type}_mapping.json')
        with open(individual_file, 'w') as f:
            json.dump(mapping, f, indent=2)
    
    # Save reverse mappings (biological_name -> index)
    reverse_mappings = {}
    for node_type, mapping in mappings_to_save.items():
        reverse_mappings[node_type] = {bio_name: str(idx) for idx, bio_name in mapping.items()}
    
    reverse_file = os.path.join(mappings_save_dir, 'reverse_biological_mappings.json')
    with open(reverse_file, 'w') as f:
        json.dump(reverse_mappings, f, indent=2)
    
    print(f"    Main mappings saved to: {main_file}")
    print(f"    Individual mappings saved to: {mappings_save_dir}/")
    print(f"    Reverse mappings saved to: {reverse_file}")

In [ ]:
def set_case_data():
    global subtypes_list, patient_features, subtype_tensor, gene_names, protein_names, cnv_names, mutation_names
    global gene_to_idx, prot_to_idx, cnv_to_idx, mutation_to_idx
    
    # Use the deterministic subtype ordering
    subtypes_list = [biological_mappings['subtype'][i] for i in sorted(biological_mappings['subtype'].keys())]
    
    # Extract patient features (excluding subtype columns)
    feature_cols = [col for col in tcga_brca_data.columns if col not in subtypes_list]
    patient_features = tcga_brca_data[feature_cols].values.astype(np.float32)
    
    # Create deterministic subtype tensor using our mapping
    subtype_labels = []
    for idx, row in subtype_data.iterrows():
        subtype_name = row.idxmax()  # Get the subtype with max value
        # Find the index in our deterministic mapping
        subtype_idx = None
        for mapped_idx, mapped_name in biological_mappings['subtype'].items():
            if mapped_name == subtype_name:
                subtype_idx = mapped_idx
                break
        subtype_labels.append(subtype_idx if subtype_idx is not None else 0)
    
    subtype_tensor = torch.tensor(subtype_labels, dtype=torch.long)
    
    # Create deterministic feature name mappings using sorted order
    gene_names = [biological_mappings['gene'][i] for i in sorted(biological_mappings['gene'].keys())]
    protein_names = [biological_mappings['protein'][i] for i in sorted(biological_mappings['protein'].keys())]
    cnv_names = [biological_mappings['cnv'][i] for i in sorted(biological_mappings['cnv'].keys())]
    mutation_names = [biological_mappings['mutation'][i] for i in sorted(biological_mappings['mutation'].keys())]
    
    # Create reverse lookup dictionaries
    gene_to_idx = {name: idx for idx, name in enumerate(gene_names)}
    prot_to_idx = {name: idx for idx, name in enumerate(protein_names)}
    cnv_to_idx = {name: idx for idx, name in enumerate(cnv_names)}
    mutation_to_idx = {name: idx for idx, name in enumerate(mutation_names)}

In [ ]:
def get_deterministic_data_matrices():
    """Get data matrices in deterministic order based on our mappings"""
    
    # Reorder columns to match our deterministic mappings
    sorted_gene_cols = sorted(gene_data.columns.tolist())
    sorted_protein_cols = sorted(protein_data.columns.tolist())
    sorted_cnv_cols = sorted(cnv_data.columns.tolist())
    sorted_mutation_cols = sorted(mutation_data.columns.tolist())
    
    gene_data_sorted = gene_data.reindex(sorted_gene_cols, axis=1)
    protein_data_sorted = protein_data.reindex(sorted_protein_cols, axis=1)
    cnv_data_sorted = cnv_data.reindex(sorted_cnv_cols, axis=1)
    mutation_data_sorted = mutation_data.reindex(sorted_mutation_cols, axis=1)
    
    return gene_data_sorted, protein_data_sorted, cnv_data_sorted, mutation_data_sorted

In [ ]:
def build_feature_network(feat_matrix, threshold=0.7):
    corr = np.corrcoef(feat_matrix) 
    idx = np.arange(corr.shape[0])
    mask = (np.abs(corr) > threshold) & (idx[:, None] != idx[None, :])
    src, dst = np.nonzero(mask)
    edge_index_np = np.vstack((src, dst))
    return torch.from_numpy(edge_index_np).long()

In [ ]:
# 1) Gene < - > Case
def metapath_gene_in_case():
    print('\tDefining "Gene in Case" Metapath...')
    gene_np = gene_data.values
    
    case_idx_list = []
    gene_idx_list = []
    
    for i in range(gene_data.shape[0]): # Number of patient (case) nodes
        nz_cols = np.nonzero(gene_np[i, :])[0]    
        for j in nz_cols:
            case_idx_list.append(i)
            gene_idx_list.append(j)
    edge_gene_in_case = torch.tensor([gene_idx_list, case_idx_list], dtype=torch.long)
    
    return edge_gene_in_case

def metapath_case_has_gene():
    print('\tDefining "Case has Gene" Metapath...')
    gene_np = gene_data.values

    case_idx_list = []
    gene_idx_list = []

    for i in range(gene_data.shape[0]):  # Number of patient (case) nodes
        nz_cols = np.nonzero(gene_np[i, :])[0]    
        for j in nz_cols:
            case_idx_list.append(i)
            gene_idx_list.append(j)

    edge_case_has_gene = torch.tensor([case_idx_list, gene_idx_list], dtype=torch.long)

    return edge_case_has_gene
    

In [ ]:
# 2) Protein < - > Case
def metapath_protein_in_case():
    print('\tDefining "Protein in Case" Metapath...')
    protein_np = protein_data.values
    
    case_idx_list = []
    protein_idx_list = []
    
    for i in range(gene_data.shape[0]): 
        nz_cols = np.nonzero(protein_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            protein_idx_list.append(j)
    
    edge_protein_in_case = torch.tensor([protein_idx_list, case_idx_list], dtype=torch.long)

    return edge_protein_in_case

def metapath_case_has_protein():
    print('\tDefining "Case has Protein" Metapath...')
    protein_np = protein_data.values
    
    case_idx_list = []
    protein_idx_list = []
    
    for i in range(gene_data.shape[0]): 
        nz_cols = np.nonzero(protein_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            protein_idx_list.append(j)
    
    edge_case_has_protein = torch.tensor([case_idx_list, protein_idx_list], dtype=torch.long)

    return edge_case_has_protein    

In [ ]:
# 3) CNV < - > Case
def metapath_cnv_in_case():
    print('\tDefining "CNV in Case" Metapath...')
    cnv_np = cnv_data.values
    
    case_idx_list = []
    cnv_idx_list = []
    
    for i in range(gene_data.shape[0]):
        nz_cols = np.nonzero(cnv_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            cnv_idx_list.append(j)
    
    edge_cnv_in_case = torch.tensor([cnv_idx_list, case_idx_list], dtype=torch.long)

    return edge_cnv_in_case

def metapath_case_has_cnv():
    print('\tDefining "Case has CNV" Metapath...')
    cnv_np = cnv_data.values
    
    case_idx_list = []
    cnv_idx_list = []
    
    for i in range(gene_data.shape[0]):
        nz_cols = np.nonzero(cnv_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            cnv_idx_list.append(j)
    
    edge_case_has_cnv = torch.tensor([case_idx_list, cnv_idx_list], dtype=torch.long)

    return edge_case_has_cnv    

In [ ]:
# 4) Mutation < - > Case
def metapath_mutation_in_case():
    print('\tDefining "Mutation in Case" Metapath...')
    mutation_np = mutation_data.values 
    
    case_idx_list = []
    mutation_idx_list = []
    
    for i in range(gene_data.shape[0]):
        nz_cols = np.nonzero(mutation_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            mutation_idx_list.append(j)
    
    edge_mutation_in_case = torch.tensor([mutation_idx_list, case_idx_list], dtype=torch.long)

    return edge_mutation_in_case

def metapath_case_has_mutation():
    print('\tDefining "Case has Mutation" Metapath...')
    mutation_np = mutation_data.values 
    
    case_idx_list = []
    mutation_idx_list = []
    
    for i in range(gene_data.shape[0]):
        nz_cols = np.nonzero(mutation_np[i, :])[0]
        for j in nz_cols:
            case_idx_list.append(i)
            mutation_idx_list.append(j)
    
    edge_case_has_mutation = torch.tensor([case_idx_list, mutation_idx_list], dtype=torch.long)

    return edge_case_has_mutation    

In [ ]:
# 5) Case < - > Case (Similar To)
def metapath_case_similar_to_case():
    print('\tDefining "Case Similar to Case" Metapath...')
    sim_matrix = cosine_similarity(patient_features)
    
    thresh = 0.8
    mask = (sim_matrix > thresh) & (sim_matrix < 1.0)
    src, dst = np.nonzero(mask) 
    
    edge_index_np = np.vstack((src, dst))  
    edge_case_similar_to_case = torch.from_numpy(edge_index_np).long()

    return edge_case_similar_to_case

In [ ]:
# 6) Case < - > Case (coexpression)
def metapath_case_coexpress_case():
    print('\tDefining "Case Coexpress Case" Metapath...')
    src, dst = metapath_gene_in_case().numpy()
    values = np.ones_like(dst, dtype=np.float32)
    A = sp.csr_matrix((values, (dst, src)), shape=(gene_data.shape[0], gene_data.shape[1]))
    
    C = A.dot(A.T).tocsr()
    
    C.setdiag(0)
    C.eliminate_zeros()
    
    case_i, case_j = C.nonzero()
    
    edge_case_to_case_coexpress = torch.from_numpy(
        np.vstack((case_i, case_j))
    ).long()

    return edge_case_to_case_coexpress

In [ ]:
# 7) Gene < - > Gene
def metapath_gene_interacts_gene():
    print('\tDefining "Gene Interacts Gene" Metapath...')
    gene_feat = gene_data.values.T
    edge_gene_gene = build_feature_network(gene_feat, threshold=0.7)

    return edge_gene_gene

In [ ]:
# 8) Protein < - > Protein
def metapath_protein_ppi_protein():
    print('\tDefining "Protein Interacts Protein" Metapath...')
    protein_feat = protein_data.values.T
    edge_ppi = build_feature_network(protein_feat, threshold=0.7)

    return edge_ppi

In [ ]:
# 9) CNV < - > CNV
def metapath_cnv_co_cnv_cnv():
    print('\tDefining "CNV Co-CNV CNV" Metapath...')
    cnv_feat = cnv_data.values.T
    edge_cnv_cnv = build_feature_network(cnv_feat, threshold=0.7)

    return edge_cnv_cnv

In [ ]:
# 10) Mutation < - > Mutation
def metapath_mutation_co_mutation_mutation():
    print('\tDefining "Mutation Co-Mutation Mutation" Metapath...')
    mutation_feat = mutation_data.values.T
    edge_mu_mu = build_feature_network(mutation_feat, threshold=0.7)

    return edge_mu_mu

In [ ]:
# 11) Gene < - > Protein
def metapath_gene_encodes_protein():
    print('\tDefining "Gene Encodes Protein" Metapath...')    
    src, dst = [], []
    for g, gi in gene_to_idx.items():
        if g in prot_to_idx:
            src.append(gi)
            dst.append(prot_to_idx[g])

    return [src, dst]

In [ ]:
# 12) Mutation < - > Gene
def metapath_mutation_in_gene_gene():
    print('\tDefining "Mutation in Gene" Metapath...')
    mu_genes = [c.split('_', 1)[1] for c in mutation_data.columns]
    src_mu, dst_mu = [], []
    for mu_idx, gene_name in enumerate(mu_genes):
        if gene_name in gene_to_idx:
            src_mu.append(mu_idx)           
            dst_mu.append(gene_to_idx[gene_name])
    
    edge_mu_to_gene = np.vstack((src_mu, dst_mu))

    return edge_mu_to_gene

In [ ]:
# 13) CNV < - > Gene
def metapath_cnv_affects_gene():
    print('\tDefining "CNV affects Gene" Metapath...')
    cnv_genes = [c.split('_', 1)[1] for c in cnv_data.columns]
    src_cnv, dst_cnv = [], []
    for cnv_idx, gene_name in enumerate(cnv_genes):
        if gene_name in gene_to_idx:
            src_cnv.append(cnv_idx)
            dst_cnv.append(gene_to_idx[gene_name])
    
    edge_cnv_to_gene = np.vstack((src_cnv, dst_cnv))

    return edge_cnv_to_gene

In [ ]:
# 14) Case < - > Subtype
def metapath_case_has_subtype_subtype():
    print('\tDefining "Case has Subtype" Metapath...')
    case2sub_s, case2sub_t = [], []
    for i, label in enumerate(tcga_brca_data[subtypes_list].values):
        subtype_idx = int(label.argmax())
        case2sub_s.append(i)
        case2sub_t.append(subtype_idx)

    return [case2sub_s, case2sub_t]

In [ ]:
# 15) Subtype < - > Gene (Characterized By)
def metapath_subtype_characterized_by_gene(top_k):
    print('\tDefining "Subtype Characterized by Gene" Metapath...')
    
    genes = gene_data.values
    src_all = []
    dst_all = []
    
    for s_idx, subtype_name in enumerate(subtypes_list):
        mask = tcga_brca_data[subtype_name].astype(bool)
        mean_gene = genes[mask].mean(axis=0)
        
        # Include top k genes if specified, otherwise include all
        if top_k is not None:
            top_genes = np.argsort(mean_gene)[-top_k:]
            src = [s_idx] * top_k
            dst = top_genes.tolist()
            return [src, dst]
        else:
            # Include all genes
            num_genes = len(mean_gene)
            src = [s_idx] * num_genes
            dst = list(range(num_genes))
        
            src_all.extend(src)
            dst_all.extend(dst)
            return [src_all, dst_all]

In [ ]:
# 15) Subtype < - > Protein (Characterized By)
def metapath_subtype_characterized_by_protein(top_k):
    print('\tDefining "Subtype Characterized by Protein" Metapath...')
    
    proteins = protein_data.values
    src_all = []
    dst_all = []
    
    for s_idx, subtype_name in enumerate(subtypes_list):
        mask = tcga_brca_data[subtype_name].astype(bool)
        mean_protein = proteins[mask].mean(axis=0)
        
        # Include top k proteins if specified, otherwise include all
        if top_k is not None:
            top_proteins = np.argsort(mean_protein)[-top_k:]
            src = [s_idx] * top_k
            dst = top_proteins.tolist()
            return [src, dst]
        else:
            # Include all proteins
            num_proteins = len(mean_protein)
            src = [s_idx] * num_proteins
            dst = list(range(num_proteins))
        
            src_all.extend(src)
            dst_all.extend(dst)
            return [src_all, dst_all]

In [ ]:
# 15) Subtype < - > CNV (Characterized By)
def metapath_subtype_characterized_by_cnv(top_k):
    print('\tDefining "Subtype Characterized by CNV" Metapath...')
    
    cnvs = cnv_data.values
    src_all = []
    dst_all = []
    
    for s_idx, subtype_name in enumerate(subtypes_list):
        mask = tcga_brca_data[subtype_name].astype(bool)
        mean_cnv = cnvs[mask].mean(axis=0)
        
        # Include top k CNVs if specified, otherwise include all
        if top_k is not None:
            top_cnvs = np.argsort(mean_cnv)[-top_k:]
            src = [s_idx] * top_k
            dst = top_cnvs.tolist()
            return [src, dst]
        else:
            # Include all CNVs
            num_cnvs = len(mean_cnv)
            src = [s_idx] * num_cnvs
            dst = list(range(num_cnvs))
        
            src_all.extend(src)
            dst_all.extend(dst)
            return [src_all, dst_all]

In [ ]:
# 15) Subtype < - > Mutation (Characterized By)
def metapath_subtype_characterized_by_mutation(top_k):
    print('\tDefining "Subtype Characterized by Mutation" Metapath...')
    
    mutations = mutation_data.values
    src_all = []
    dst_all = []
    
    for s_idx, subtype_name in enumerate(subtypes_list):
        mask = tcga_brca_data[subtype_name].astype(bool)
        mean_mutation = mutations[mask].mean(axis=0)
        
        # Include top k mutations if specified, otherwise include all
        if top_k is not None:
            top_mutations = np.argsort(mean_mutation)[-top_k:]
            src = [s_idx] * top_k
            dst = top_mutations.tolist()
            return [src, dst]
        else:
            # Include all mutations
            num_mutations = len(mean_mutation)
            src = [s_idx] * num_mutations
            dst = list(range(num_mutations))
        
            src_all.extend(src)
            dst_all.extend(dst)
            return [src_all, dst_all]

In [ ]:
heteroData = HeteroData()

def hetrograph_constructor(top_k):
    print('\nDefining Case and Subtype Nodes...')
    heteroData['case'].x = patient_features   
    heteroData['case'].y = subtype_tensor 

    heteroData['subtype'].x = torch.eye(len(subtypes_list), dtype=torch.float)

    print('Defining Multi-Omic Nodes...')
    heteroData['gene'].x = torch.ones((gene_data.shape[1], 1)) * 0. 
    heteroData['protein'].x = torch.ones((protein_data.shape[1], 1)) * 0.
    heteroData['cnv'].x = torch.ones((cnv_data.shape[1], 1)) * 0.
    heteroData['mutation'].x = torch.ones((mutation_data.shape[1], 1)) * 0.

    print('\nDefining Multi-Omics - Pateint Relations...')
    heteroData['gene', 'in', 'case'].edge_index = metapath_gene_in_case()
    heteroData['protein', 'in', 'case'].edge_index = metapath_protein_in_case()
    heteroData['cnv', 'in', 'case'].edge_index = metapath_cnv_in_case()
    heteroData['mutation', 'in', 'case'].edge_index = metapath_mutation_in_case()

    heteroData['case', 'has', 'gene'].edge_index = metapath_case_has_gene()
    heteroData['case', 'has', 'protein'].edge_index = metapath_case_has_protein()
    heteroData['case', 'has', 'cnv'].edge_index = metapath_case_has_cnv()
    heteroData['case', 'has', 'mutation'].edge_index = metapath_case_has_mutation()

    print('\nDefining Patient - Patient Similarity Edges...')
    heteroData['case', 'similar_to', 'case'].edge_index = metapath_case_similar_to_case()
    heteroData['case', 'coexpr_with', 'case'].edge_index = metapath_case_coexpress_case()

    print('\nDefining Intra-Omic Feature Networks...')
    heteroData['gene', 'interacts', 'gene'].edge_index = metapath_gene_interacts_gene()
    heteroData['protein', 'ppi', 'protein'].edge_index = metapath_protein_ppi_protein()
    heteroData['cnv', 'co_cnv', 'cnv'].edge_index = metapath_cnv_co_cnv_cnv()
    heteroData['mutation', 'co_mutation', 'mutation'].edge_index = metapath_mutation_co_mutation_mutation()

    print('\nDefining Cross-Omic Bridges...')    
    heteroData['gene', 'encodes', 'protein'].edge_index = torch.tensor(np.array(metapath_gene_encodes_protein()), dtype=torch.long)
    # heteroData['protein', 'encoded_by', 'gene'].edge_index = torch.tensor([dst, src], dtype=torch.long)
    heteroData['mutation', 'in_gene', 'gene'].edge_index = torch.from_numpy(metapath_mutation_in_gene_gene()).long()
    heteroData['cnv', 'affects', 'gene'].edge_index = torch.from_numpy(metapath_cnv_affects_gene()).long()

    print('\nDefining Subtype Anchor Nodes Relations...')    
    heteroData['case', 'has_subtype', 'subtype'].edge_index = torch.tensor(metapath_case_has_subtype_subtype(), dtype=torch.long)
    heteroData['subtype', 'characterized_by', 'gene'].edge_index = torch.cat([
        heteroData['subtype', 'characterized_by', 'gene'].edge_index,
        torch.tensor([src, dst], dtype=torch.long)
    ], dim=1) if 'characterized_by' in heteroData.edge_types else torch.tensor(metapath_subtype_characterized_by_gene(top_k), dtype=torch.long)
    heteroData['subtype', 'characterized_by', 'protein'].edge_index = torch.cat([
        heteroData['subtype', 'characterized_by', 'protein'].edge_index,
        torch.tensor([src, dst], dtype=torch.long)
    ], dim=1) if 'characterized_by' in heteroData.edge_types else torch.tensor(metapath_subtype_characterized_by_protein(top_k), dtype=torch.long)
    heteroData['subtype', 'characterized_by', 'cnv'].edge_index = torch.cat([
        heteroData['subtype', 'characterized_by', 'cnv'].edge_index,
        torch.tensor([src, dst], dtype=torch.long)
    ], dim=1) if 'characterized_by' in heteroData.edge_types else torch.tensor(metapath_subtype_characterized_by_cnv(top_k), dtype=torch.long)
    heteroData['subtype', 'characterized_by', 'mutation'].edge_index = torch.cat([
        heteroData['subtype', 'characterized_by', 'mutation'].edge_index,
        torch.tensor([src, dst], dtype=torch.long)
    ], dim=1) if 'characterized_by' in heteroData.edge_types else torch.tensor(metapath_subtype_characterized_by_mutation(top_k), dtype=torch.long)    


In [ ]:
def construct_hetrograph(preprocessed_df, top_k=None):
    set_source_datasets(preprocessed_df)
    create_persistent_mappings()
    set_case_data()
    hetrograph_constructor(top_k)
    
    print("\n",heteroData)
    print("\nNode sets:", heteroData.node_types)
    print("\nEdge sets:", heteroData.edge_types)

    return heteroData